# CS336 Spring 2026 - Special Tokens
----
> Special-token 应该是整个字符串是一个 token, 不能让普通 regex 或 BPE 决定怎么拆.
> 
> 本节最重要的思想是: Special-token 检测是在 pre-tokenization 之前.

## 1. 先恢复上一节的 pre-tokenizer

In [1]:
import regex
from collections import Counter


GPT2_PRETOKEN_PATTERN = (
    r"""'(?:[sdmt]|ll|ve|re)"""
    r"""| ?\p{L}+"""
    r"""| ?\p{N}+"""
    r"""| ?[^\s\p{L}\p{N}]+"""
    r"""|\s+(?!\S)"""
    r"""|\s+"""
)


def pretokenize(text: str) -> list[str]:

    return [match.group(0) for match in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


## 2. 错误实验: 将 Special Token 作为普通文本
本来希望 `<|endoftext|>` 是一个完整的 token, 但是普通的 pre-tokenizer 将其拆开. 若继续使用 BPE, 这些会继续进入下一部分, 但显然是不对的.

In [2]:
text = "Hello<|endoftext|>world"

print(pretokenize(text))

['Hello', '<|', 'endoftext', '|>', 'world']


## 3. Special Token 的核心性质: Atomic
若定义了:
```python
special_tokens = ["<|endoftext|>"]
```
那么下面所有的位置:
```text
<|endoftext|>

hello<|endoftext|>

<|endoftext|>hello

hello<|endoftext|>world
```
都必须把 `<|endoftext|>` 识别成一个整体. 这就是所谓的 atomic token.
> tokenizer 的普通逻辑不能继续深入 special token 内部.

## 4. 正确的处理顺序
正确顺序为:
```text
whole text
    ↓
先检测 special tokens
    ↓
切分成 special spans / ordinary spans
    ↓
只有 ordinary spans 进入 regex
```
即 `Hello<|endoftext|>world`, 先切:
```text
ORDINARY  "Hello"

SPECIAL   "<|endoftext|>"

ORDINARY  "world"
```
然后只有 `ORDERINARY` 进入普通的 pre-tokenization.

## 5. 构造 Special Token Pattern
我们需要一个 pattern 来寻找 special tokens. 先考虑:
```python
special_tokens = ["<|endoftext|>"]
```
注意这里的 `< | >` 在 regex 中具有特殊意义. 因此需要:
```python
regex.escape(...)
```

In [ ]:
def build_special_pattern(special_tokens: list[str]) -> regex.Pattern | None:
    # 构造 Special Token Pattern
    if not special_tokens:
        return None

    ordered = sorted(
        set(special_tokens), key=lambda token: (-len(token), token)
    )  # 去重并按长度降序排序
    alternatives = "|".join(regex.escape(token) for token in ordered)

    return regex.compile(f"({alternatives})")

### Q: 为什么上述要进行长度降序排序?
A: 正则中的`|` 通常从左到右匹配, 一旦匹配成功就停止. 如果按较短的 token 排在前面, 它可能会先匹配, 导致较长的 token 被错误的拆开.

 例如 `<|endof` 和 `<|endoftext|>`, 若 `<|endof` 在前, 文本 `<|endoftext|>` 会被拆成 `<endof` + `text|>`.

In [4]:
pattern = build_special_pattern(["<|endoftext|>"])

print(pattern.pattern)

(<\|endoftext\|>)


可以看到 `|` 被 escape 成了 `\|`

### Q: 为什么使用了 capturing group?
A: 因为是在后续希望 `regex.split(...)` 不仅返回 special token 两边的文本, 还把 special token 本身保存下来.

例如, 对 `Hello<|endoftext|>World`, 希望
```text
Hello

<|endoftext|>

World
```
而不是将 special token 删除掉.

## 6. 隔离 Ordinary Span 和 Special Span

In [5]:
def split_around_special_tokens(
    text: str, special_tokens: list[str]
) -> list[tuple[bool, str]]:
    pattern = build_special_pattern(special_tokens)

    if pattern is None:
        return [(False, text)] if text else []

    special_set = set(special_tokens)
    output: list[tuple[bool, str]] = []

    for piece in pattern.split(text):
        if piece == "":
            continue

        output.append((piece in special_set, piece))

    return output

In [6]:
# 进行测试
print(split_around_special_tokens("Hello<|endoftext|>world", ["<|endoftext|>"]))

[(False, 'Hello'), (True, '<|endoftext|>'), (False, 'world')]
